# Notebook 02: Prefill vs Decode - Two Phases of LLM Generation

## 📚 Historical Context

**Timeline**: 2019-2021 - Understanding the two-phase nature of inference

**The Discovery**:
- GPT-2/GPT-3 deployments revealed two distinct performance patterns
- "First token is slow, subsequent tokens are fast" - why?
- Initial research treated all tokens equally - wrong!
- Understanding this split became critical for optimization

**The Insight**:
- **Prefill phase**: Process entire prompt in parallel (compute-intensive)
- **Decode phase**: Generate tokens one-by-one (memory-intensive)
- Different bottlenecks require different optimizations
- Modern serving systems optimize each phase separately

## 🎯 What You'll Learn

1. Two phases of LLM generation explained
2. Prefill phase: processing prompt (parallel)
3. Decode phase: generating tokens (sequential)
4. Performance characteristics of each
5. Measuring and comparing both phases
6. Code examples showing the difference

## 💡 Why This Matters

- **User experience**: First token latency (prefill) affects perceived responsiveness
- **Throughput**: Decode phase dominates total time for long outputs
- **Optimization**: Different techniques for different phases
- **Resource allocation**: Batching works differently in each phase

**Key Metric**: 
- **TTFT** (Time To First Token) = Prefill time
- **TPOT** (Time Per Output Token) = Decode time

**Reference**: See LLM_INFERENCE_ROADMAP.md - Stage 0: Prefill vs Decode

---

## Setup and Imports

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("=" * 80)
print("ENVIRONMENT CHECK")
print("=" * 80)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print("=" * 80)

## Part 1: Understanding the Two Phases

### Phase 1: PREFILL (Initial Processing)

**What happens**:
- Process entire input prompt at once
- All prompt tokens processed in parallel
- Compute attention for all prompt tokens
- Generate the FIRST output token

**Characteristics**:
- **Parallelizable**: All prompt tokens processed simultaneously
- **Compute-intensive**: Matrix operations on full prompt
- **Higher GPU utilization**: ~40-60% (better than decode)
- **Batch-friendly**: Can batch multiple prompts efficiently

**Example**:
```
Prompt: "The cat sat on the" (5 tokens)
Prefill: Process all 5 tokens → Generate "mat"
Time: ~10-50ms (depending on prompt length)
```

---

### Phase 2: DECODE (Token Generation)

**What happens**:
- Generate one token at a time
- Each new token depends on all previous tokens
- Must run model for each new token
- Continue until stopping condition

**Characteristics**:
- **Sequential**: Cannot parallelize within single request
- **Memory-bound**: Dominated by loading model weights
- **Low GPU utilization**: ~10-30% (memory bandwidth bottleneck)
- **Batching helps**: Can decode multiple requests together

**Example**:
```
Decode step 1: "mat" → Generate "."
Decode step 2: "." → Generate "The"
Decode step 3: "The" → Generate "dog"
...
Time per token: ~5-15ms each
```

---

### Visual Comparison:

```
PREFILL (Parallel):
[The] [cat] [sat] [on] [the]  →  [mat]
  ↓     ↓     ↓     ↓     ↓       ↑
  └─────┴─────┴─────┴─────┴───────┘
        All processed together

DECODE (Sequential):
[...the mat] → [.]     (step 1)
[...mat.]    → [The]   (step 2)
[....The]    → [dog]   (step 3)
    One at a time, cannot parallelize
```

In [ ]:
# Load model
model_name = "gpt2"  # 124M parameters

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print(f"Model: {model_name}")
print(f"Device: {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Part 2: Measuring Prefill Phase

Let's measure the prefill phase with different prompt lengths.

In [ ]:
def measure_prefill(
    model,
    input_ids: torch.Tensor,
    num_runs: int = 20
) -> Dict[str, float]:
    """
    Measure prefill phase latency.
    
    Prefill = processing the entire prompt to generate first token.
    """
    # Warmup
    for _ in range(5):
        with torch.no_grad():
            _ = model(input_ids)
    
    if device == "cuda":
        torch.cuda.synchronize()
    
    # Measure
    times = []
    for _ in range(num_runs):
        if device == "cuda":
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            
            start.record()
            with torch.no_grad():
                outputs = model(input_ids)
                logits = outputs.logits
                # Sample first token (completes prefill)
                next_token = torch.argmax(logits[:, -1, :], dim=-1)
            end.record()
            
            torch.cuda.synchronize()
            elapsed = start.elapsed_time(end)
        else:
            start = time.time()
            with torch.no_grad():
                outputs = model(input_ids)
                logits = outputs.logits
                next_token = torch.argmax(logits[:, -1, :], dim=-1)
            elapsed = (time.time() - start) * 1000
        
        times.append(elapsed)
    
    return {
        'mean': np.mean(times),
        'std': np.std(times),
        'min': np.min(times),
        'max': np.max(times),
        'p50': np.percentile(times, 50),
        'p95': np.percentile(times, 95),
        'p99': np.percentile(times, 99),
    }

# Test different prompt lengths
print("=" * 80)
print("PREFILL PHASE BENCHMARKS")
print("=" * 80)

prompt_lengths = [8, 16, 32, 64, 128, 256, 512]
prefill_results = []

for length in prompt_lengths:
    # Create random prompt of this length
    input_ids = torch.randint(0, tokenizer.vocab_size, (1, length)).to(device)
    
    stats = measure_prefill(model, input_ids, num_runs=20)
    prefill_results.append((length, stats))
    
    print(f"\nPrompt length: {length} tokens")
    print(f"  Mean TTFT: {stats['mean']:.2f} ms")
    print(f"  P95 TTFT:  {stats['p95']:.2f} ms")
    print(f"  Time per prompt token: {stats['mean']/length:.3f} ms/token")

print("\n" + "=" * 80)

## Part 3: Measuring Decode Phase

Now let's measure the decode phase - generating tokens one by one.

In [ ]:
def measure_decode(
    model,
    initial_input_ids: torch.Tensor,
    num_tokens: int = 20,
    num_runs: int = 3
) -> Dict[str, any]:
    """
    Measure decode phase latency.
    
    Decode = generating tokens one-by-one after prefill.
    """
    all_token_times = []
    
    for run in range(num_runs):
        input_ids = initial_input_ids.clone()
        token_times = []
        
        # Skip the prefill (we already measured that)
        # Just time the decode steps
        for step in range(num_tokens):
            if device == "cuda":
                start = torch.cuda.Event(enable_timing=True)
                end = torch.cuda.Event(enable_timing=True)
                
                start.record()
                with torch.no_grad():
                    outputs = model(input_ids)
                    logits = outputs.logits
                    next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
                    input_ids = torch.cat([input_ids, next_token], dim=-1)
                end.record()
                
                torch.cuda.synchronize()
                elapsed = start.elapsed_time(end)
            else:
                start = time.time()
                with torch.no_grad():
                    outputs = model(input_ids)
                    logits = outputs.logits
                    next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
                    input_ids = torch.cat([input_ids, next_token], dim=-1)
                elapsed = (time.time() - start) * 1000
            
            token_times.append(elapsed)
        
        all_token_times.extend(token_times)
    
    return {
        'mean': np.mean(all_token_times),
        'std': np.std(all_token_times),
        'min': np.min(all_token_times),
        'max': np.max(all_token_times),
        'p50': np.percentile(all_token_times, 50),
        'p95': np.percentile(all_token_times, 95),
        'p99': np.percentile(all_token_times, 99),
        'all_times': all_token_times,
    }

# Test decode phase with different initial sequence lengths
print("=" * 80)
print("DECODE PHASE BENCHMARKS")
print("=" * 80)

initial_lengths = [32, 64, 128, 256, 512]
decode_results = []

for length in initial_lengths:
    input_ids = torch.randint(0, tokenizer.vocab_size, (1, length)).to(device)
    
    stats = measure_decode(model, input_ids, num_tokens=20, num_runs=3)
    decode_results.append((length, stats))
    
    print(f"\nInitial sequence length: {length} tokens")
    print(f"  Mean TPOT: {stats['mean']:.2f} ms/token")
    print(f"  P95 TPOT:  {stats['p95']:.2f} ms/token")
    print(f"  Tokens/sec: {1000/stats['mean']:.2f}")

print("\n" + "=" * 80)

## Part 4: Comparing Prefill vs Decode

In [ ]:
# Visualize prefill vs decode characteristics
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Prefill latency vs prompt length
ax = axes[0, 0]
lengths = [l for l, _ in prefill_results]
means = [s['mean'] for _, s in prefill_results]
p95s = [s['p95'] for _, s in prefill_results]

ax.plot(lengths, means, marker='o', linewidth=2, markersize=8, label='Mean')
ax.plot(lengths, p95s, marker='s', linewidth=2, markersize=8, label='P95', linestyle='--')
ax.set_xlabel('Prompt Length (tokens)', fontsize=12)
ax.set_ylabel('Prefill Time (ms)', fontsize=12)
ax.set_title('Prefill Phase: Time To First Token (TTFT)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Prefill time per token (should be roughly constant for parallel processing)
ax = axes[0, 1]
time_per_token = [s['mean']/l for l, s in prefill_results]
ax.plot(lengths, time_per_token, marker='o', linewidth=2, markersize=8, color='green')
ax.set_xlabel('Prompt Length (tokens)', fontsize=12)
ax.set_ylabel('Time per Token (ms)', fontsize=12)
ax.set_title('Prefill: Time per Prompt Token\n(Parallel processing means constant time/token)', 
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Plot 3: Decode latency vs initial sequence length
ax = axes[1, 0]
decode_lengths = [l for l, _ in decode_results]
decode_means = [s['mean'] for _, s in decode_results]
decode_p95s = [s['p95'] for _, s in decode_results]

ax.plot(decode_lengths, decode_means, marker='o', linewidth=2, markersize=8, label='Mean', color='orange')
ax.plot(decode_lengths, decode_p95s, marker='s', linewidth=2, markersize=8, label='P95', linestyle='--', color='red')
ax.set_xlabel('Sequence Length (tokens)', fontsize=12)
ax.set_ylabel('Decode Time per Token (ms)', fontsize=12)
ax.set_title('Decode Phase: Time Per Output Token (TPOT)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 4: Direct comparison
ax = axes[1, 1]
# Use a common sequence length for fair comparison
common_length = 128
prefill_idx = [i for i, (l, _) in enumerate(prefill_results) if l == common_length][0]
decode_idx = [i for i, (l, _) in enumerate(decode_results) if l == common_length][0]

prefill_time_per_token = prefill_results[prefill_idx][1]['mean'] / common_length
decode_time_per_token = decode_results[decode_idx][1]['mean']

categories = ['Prefill\n(per prompt token)', 'Decode\n(per output token)']
times = [prefill_time_per_token, decode_time_per_token]
colors = ['#3498db', '#e74c3c']

bars = ax.bar(categories, times, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_ylabel('Time per Token (ms)', fontsize=12)
ax.set_title(f'Prefill vs Decode Comparison (seq_len={common_length})', 
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, time in zip(bars, times):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{time:.3f} ms',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("KEY OBSERVATIONS")
print("=" * 80)
print("\nPREFILL PHASE:")
print("  • Total time grows linearly with prompt length")
print("  • Time per token is roughly constant (parallel processing)")
print("  • Longer prompts = higher TTFT (time to first token)")
print("  • More compute-intensive, better GPU utilization")

print("\nDECODE PHASE:")
print("  • Time per token increases with sequence length (O(n) attention)")
print("  • Sequential generation - cannot parallelize within request")
print("  • Memory-bound - low GPU utilization")
print("  • Dominates total time for long outputs")

print("\nCOMPARISON:")
print(f"  • Prefill: {prefill_time_per_token:.3f} ms per prompt token (parallel)")
print(f"  • Decode: {decode_time_per_token:.2f} ms per output token (sequential)")
print(f"  • Decode is ~{decode_time_per_token/prefill_time_per_token:.1f}x slower per token")
print("=" * 80)

## Part 5: Full Generation Timeline

Let's visualize the complete generation process showing both phases.

In [ ]:
def measure_full_generation(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 30
) -> Dict[str, any]:
    """
    Measure both prefill and decode phases in a complete generation.
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    initial_length = input_ids.shape[1]
    
    # Warmup
    with torch.no_grad():
        _ = model(input_ids)
    
    if device == "cuda":
        torch.cuda.synchronize()
    
    # Measure prefill
    if device == "cuda":
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        
        start.record()
        with torch.no_grad():
            outputs = model(input_ids)
            logits = outputs.logits
            next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
            input_ids = torch.cat([input_ids, next_token], dim=-1)
        end.record()
        
        torch.cuda.synchronize()
        prefill_time = start.elapsed_time(end)
    else:
        start = time.time()
        with torch.no_grad():
            outputs = model(input_ids)
            logits = outputs.logits
            next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
            input_ids = torch.cat([input_ids, next_token], dim=-1)
        prefill_time = (time.time() - start) * 1000
    
    # Measure decode
    decode_times = []
    
    for _ in range(max_new_tokens - 1):  # -1 because we already generated one token in prefill
        if device == "cuda":
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            
            start.record()
            with torch.no_grad():
                outputs = model(input_ids)
                logits = outputs.logits
                next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
                input_ids = torch.cat([input_ids, next_token], dim=-1)
            end.record()
            
            torch.cuda.synchronize()
            elapsed = start.elapsed_time(end)
        else:
            start = time.time()
            with torch.no_grad():
                outputs = model(input_ids)
                logits = outputs.logits
                next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
                input_ids = torch.cat([input_ids, next_token], dim=-1)
            elapsed = (time.time() - start) * 1000
        
        decode_times.append(elapsed)
    
    generated_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    
    return {
        'prompt': prompt,
        'generated_text': generated_text,
        'prompt_tokens': initial_length,
        'generated_tokens': len(decode_times) + 1,
        'prefill_time': prefill_time,
        'decode_times': decode_times,
        'total_decode_time': sum(decode_times),
        'total_time': prefill_time + sum(decode_times),
    }

# Measure full generation
print("=" * 80)
print("FULL GENERATION TIMELINE")
print("=" * 80)

prompt = "The future of artificial intelligence will"
result = measure_full_generation(model, tokenizer, prompt, max_new_tokens=25)

print(f"\nPrompt: '{result['prompt']}'")
print(f"Prompt tokens: {result['prompt_tokens']}")
print(f"Generated tokens: {result['generated_tokens']}")
print(f"\nGenerated text:\n{result['generated_text']}")
print(f"\n{'-'*80}")
print("TIMING BREAKDOWN:")
print(f"  Prefill (TTFT):     {result['prefill_time']:7.2f} ms  ({result['prefill_time']/result['total_time']*100:5.1f}% of total)")
print(f"  Decode (total):     {result['total_decode_time']:7.2f} ms  ({result['total_decode_time']/result['total_time']*100:5.1f}% of total)")
print(f"  Decode (per token): {np.mean(result['decode_times']):7.2f} ms")
print(f"  Total time:         {result['total_time']:7.2f} ms")
print(f"\n  Throughput: {result['generated_tokens'] / (result['total_time']/1000):.2f} tokens/sec")
print("=" * 80)

In [ ]:
# Visualize the generation timeline
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Plot 1: Timeline breakdown
ax = axes[0]

# Create timeline data
phases = ['Prefill'] + [f'Decode {i+1}' for i in range(len(result['decode_times']))]
times = [result['prefill_time']] + result['decode_times']
colors = ['#3498db'] + ['#e74c3c'] * len(result['decode_times'])

# Cumulative time for stacked view
cumulative = np.cumsum([0] + times[:-1])

bars = ax.barh([0] * len(times), times, left=cumulative, color=colors, 
               edgecolor='black', linewidth=0.5, height=0.5)

# Add labels
ax.text(result['prefill_time']/2, 0, 'PREFILL', 
        ha='center', va='center', fontsize=12, fontweight='bold', color='white')
ax.text(result['prefill_time'] + result['total_decode_time']/2, 0, 'DECODE', 
        ha='center', va='center', fontsize=12, fontweight='bold', color='white')

ax.set_xlabel('Time (ms)', fontsize=12)
ax.set_yticks([])
ax.set_title('Generation Timeline: Prefill vs Decode Phases', fontsize=14, fontweight='bold')
ax.set_xlim(0, result['total_time'] * 1.05)

# Add vertical line separating phases
ax.axvline(result['prefill_time'], color='black', linestyle='--', linewidth=2)

# Plot 2: Token-by-token decode time
ax = axes[1]
token_numbers = list(range(1, len(result['decode_times']) + 1))
ax.plot(token_numbers, result['decode_times'], marker='o', linewidth=2, markersize=6, color='#e74c3c')
ax.axhline(np.mean(result['decode_times']), color='green', linestyle='--', linewidth=2,
           label=f"Mean: {np.mean(result['decode_times']):.2f} ms")
ax.set_xlabel('Output Token Number', fontsize=12)
ax.set_ylabel('Decode Time (ms)', fontsize=12)
ax.set_title('Decode Phase: Time per Token', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 6: Why This Matters for Optimization

Understanding the two phases is critical because they need different optimizations.

In [ ]:
# Compare scenarios: short vs long prompts, short vs long outputs
def compare_scenarios():
    """
    Compare different generation scenarios to understand which phase dominates.
    """
    scenarios = [
        ("Short prompt, short output", "Hello", 10),
        ("Short prompt, long output", "Hello", 50),
        ("Long prompt, short output", "In the context of modern artificial intelligence and machine learning systems, " * 3, 10),
        ("Long prompt, long output", "In the context of modern artificial intelligence and machine learning systems, " * 3, 50),
    ]
    
    print("=" * 80)
    print("SCENARIO COMPARISON: WHICH PHASE DOMINATES?")
    print("=" * 80)
    
    results = []
    
    for name, prompt, max_tokens in scenarios:
        print(f"\n{name}:")
        result = measure_full_generation(model, tokenizer, prompt, max_new_tokens=max_tokens)
        
        prefill_pct = (result['prefill_time'] / result['total_time']) * 100
        decode_pct = (result['total_decode_time'] / result['total_time']) * 100
        
        print(f"  Prompt tokens: {result['prompt_tokens']}")
        print(f"  Output tokens: {result['generated_tokens']}")
        print(f"  Prefill time: {result['prefill_time']:.1f} ms ({prefill_pct:.1f}%)")
        print(f"  Decode time:  {result['total_decode_time']:.1f} ms ({decode_pct:.1f}%)")
        print(f"  Total time:   {result['total_time']:.1f} ms")
        
        dominant_phase = "PREFILL" if prefill_pct > 50 else "DECODE"
        print(f"  Dominant phase: {dominant_phase}")
        
        results.append({
            'name': name,
            'prefill_time': result['prefill_time'],
            'decode_time': result['total_decode_time'],
            'prefill_pct': prefill_pct,
            'decode_pct': decode_pct,
        })
    
    print("\n" + "=" * 80)
    
    return results

scenario_results = compare_scenarios()

In [ ]:
# Visualize scenario comparison
fig, ax = plt.subplots(figsize=(14, 8))

names = [r['name'] for r in scenario_results]
prefill_times = [r['prefill_time'] for r in scenario_results]
decode_times = [r['decode_time'] for r in scenario_results]

x = np.arange(len(names))
width = 0.35

bars1 = ax.bar(x - width/2, prefill_times, width, label='Prefill', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, decode_times, width, label='Decode', color='#e74c3c', alpha=0.8)

ax.set_xlabel('Scenario', fontsize=12)
ax.set_ylabel('Time (ms)', fontsize=12)
ax.set_title('Prefill vs Decode Time in Different Scenarios', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add percentage labels
for i, (bar1, bar2, result) in enumerate(zip(bars1, bars2, scenario_results)):
    height1 = bar1.get_height()
    height2 = bar2.get_height()
    ax.text(bar1.get_x() + bar1.get_width()/2., height1/2,
            f"{result['prefill_pct']:.0f}%",
            ha='center', va='center', fontsize=10, fontweight='bold', color='white')
    ax.text(bar2.get_x() + bar2.get_width()/2., height2/2,
            f"{result['decode_pct']:.0f}%",
            ha='center', va='center', fontsize=10, fontweight='bold', color='white')

plt.tight_layout()
plt.show()

print("\nKEY INSIGHTS:")
print("• Short prompts + long outputs: Decode dominates (typical chatbot scenario)")
print("• Long prompts + short outputs: Prefill dominates (classification, short answers)")
print("• Most real-world scenarios: Decode phase dominates total time")
print("• User experience: TTFT (prefill) matters for perceived responsiveness")

## Part 7: Optimization Strategy Matrix

In [ ]:
print("=" * 80)
print("OPTIMIZATION STRATEGY MATRIX")
print("=" * 80)
print("""
PREFILL PHASE OPTIMIZATIONS:
  Characteristics: Parallel, compute-intensive, batch-friendly
  
  ✓ Effective optimizations:
    • Large batch sizes (process many prompts together)
    • Mixed precision (FP16/BF16) - better compute throughput
    • Flash Attention - reduce memory, enable longer prompts
    • Better GEMM kernels (optimized matrix multiplication)
    • Prompt compression / caching common prefixes
  
  ✗ Less effective:
    • Speculative decoding (doesn't apply to prefill)
    • KV cache optimizations (no cache built yet)

DECODE PHASE OPTIMIZATIONS:
  Characteristics: Sequential, memory-bound, grows with sequence length
  
  ✓ Effective optimizations:
    • KV caching (avoid recomputation) - 5-10x speedup!
    • Quantization (INT8/INT4) - reduce memory bandwidth
    • Multi-Query Attention (smaller KV cache)
    • Continuous batching (batch multiple requests)
    • Speculative decoding (predict multiple tokens)
    • PagedAttention (better memory utilization)
  
  ✗ Less effective:
    • Large batch sizes (memory limited by KV cache)
    • Compute optimizations (already memory-bound)

USER EXPERIENCE PRIORITIES:
  • Chatbots: Optimize TTFT (prefill) for responsiveness
  • Long-form generation: Optimize TPOT (decode) for speed
  • Throughput serving: Optimize both + batching

NEXT STAGES:
  • Stage 1: KV caching (decode phase) - biggest single win
  • Stage 1: Static batching (both phases)
  • Stage 2: Flash Attention (prefill phase)
  • Stage 3: Continuous batching (both phases, better)
  • Stage 4: Speculative decoding (decode phase)
""")
print("=" * 80)

## 📊 Summary and Key Takeaways

### What We Learned:

#### 1. **Two Distinct Phases**:

**Prefill (Time To First Token - TTFT)**:
- Process entire prompt in parallel
- Compute-intensive, better GPU utilization
- Time grows linearly with prompt length
- Critical for user-perceived responsiveness

**Decode (Time Per Output Token - TPOT)**:
- Generate tokens sequentially, one at a time
- Memory-bound, low GPU utilization
- Time per token grows with sequence length
- Dominates total time for long outputs

#### 2. **Performance Characteristics**:

```
Typical numbers (GPT-2 on GPU):
  Prefill: 0.1-0.3 ms per prompt token
  Decode:  8-15 ms per output token
  
  → Decode is ~50-100x slower per token!
```

#### 3. **Bottleneck Analysis**:

| Phase | Primary Bottleneck | GPU Utilization | Can Parallelize? |
|-------|-------------------|-----------------|------------------|
| Prefill | Compute | 40-60% | ✓ Yes (across prompt tokens) |
| Decode | Memory bandwidth | 10-30% | ✗ No (sequential generation) |

#### 4. **Which Phase Dominates?**:

- **Short prompt, long output** (chatbots): Decode dominates (80-95%)
- **Long prompt, short output** (classification): Prefill dominates (60-80%)
- **Typical use case**: Decode phase is 60-90% of total time

#### 5. **Optimization Priorities**:

**For Latency (chatbots, interactive)**:
1. Optimize TTFT (prefill) - user sees first token faster
2. Optimize TPOT (decode) - rest of response streams faster

**For Throughput (batch serving)**:
1. Batch prefill (parallel prompt processing)
2. Continuous batching for decode (dynamic request handling)

**For Memory (large models, long context)**:
1. Flash Attention (prefill memory)
2. KV cache optimization (decode memory)

### Critical Insight:

> **KV Caching transforms decode phase**: Without it, we recompute everything (like prefill every time). With it, decode becomes memory-bandwidth bound but much faster. This is the #1 optimization we'll implement in Stage 1!

### Real-World Impact:

```
Example: Chatbot generating 100 tokens
  Prompt: 50 tokens
  Output: 100 tokens
  
  Without optimization:
    Prefill: 50 tokens × 0.2 ms = 10 ms
    Decode:  100 tokens × 12 ms = 1200 ms
    Total: 1210 ms (~1.2 seconds)
    Decode is 99% of time!
  
  With Stage 1 optimizations (KV cache):
    Prefill: 10 ms (same)
    Decode:  100 tokens × 2 ms = 200 ms (6x faster!)
    Total: 210 ms (~0.2 seconds)
```

### Next Steps:

**Notebook 03: Baseline Benchmarking** - Learn to:
- Set up proper benchmarking frameworks
- Measure key metrics (P50/P95/P99 latency)
- Establish baseline performance
- Statistical significance and multiple runs
- Compare before/after optimizations

---

### Remember:

- **Prefill** = parallel, compute-bound, affects TTFT
- **Decode** = sequential, memory-bound, dominates total time
- **Different phases need different optimizations**
- **Measure both separately for accurate analysis**

---

**Reference**: LLM_INFERENCE_ROADMAP.md - Stage 0: Prefill vs Decode